# Function Setup

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import requests
import re
import json
import os
import geopandas as gpd
import pickle
import hashlib
from google import genai

In [ ]:
data_dir = Path(r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\MedianIncome_2011-2024_County")
DEFAULT_COUNTY_GEOGRAPHY_PATH = Path(
    r"C:\Users\samoj\Downloads\SCIL_Data\US_Counties_shp\tl_2025_us_county.shp"
)

In [ ]:
def get_year_from_filename(path):
    match = re.search(r"(\d{4})", path.name)
    if match:
        return int(match.group(1))
    return None


def get_table_from_filename(path):
    match = re.search(r"\.(B\d+|S\d+)-Data", path.name)
    if match:
        return match.group(1)
    return None


def clean_numeric_value(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x in {"-", "N", "(X)", "***", "****", ""}:
        return pd.NA

    x = x.replace(",", "").replace("$", "").replace("+", "")

    try:
        return float(x)
    except ValueError:
        return pd.NA


def county_fips_from_geo_id(series):
    # Example county GEO_ID: 0500000US01001 -> 01001
    return series.astype(str).str.extract(r"US(\d{5})")[0]

In [ ]:
def get_stable_columns(report):
    good = report[report["status"] == "OK"].copy()

    column_sets = []

    for _, row in good.iterrows():
        cols = set(row["estimate_columns"])
        if cols:
            column_sets.append(cols)

    if not column_sets:
        return []

    return sorted(set.intersection(*column_sets))


def normalize_label(label):
    if label is None:
        return ""

    label = str(label).strip().lower()

    # ACS labels sometimes add trailing colons to category names in newer years.
    # Example: "total:" vs "total", "civilian labor force:" vs "civilian labor force"
    label = label.replace(":", "")

    label = label.replace("--", "!!")
    label = " ".join(label.split())

    return label

def label_signature(label):
    label = normalize_label(label)

    # Remove year-specific inflation-adjusted dollar text
    label = re.sub(
        r"\(in \d{4} inflation-adjusted dollars\)",
        "(in inflation-adjusted dollars)",
        label
    )

    parts = [p.strip() for p in label.split("!!") if p.strip()]

    ignore = {
        "estimate",
        "total",
        "number",
        "margin of error"
    }

    parts = [p for p in parts if p not in ignore]

    signature = "!!".join(parts)

    # Normalize common ACS housing tenure wording differences
    signature = signature.replace(
        "occupied housing units!!occupied housing units",
        "occupied housing units"
    )

    signature = signature.replace(
        "occupied housing units!!renter occupied",
        "renter occupied"
    )

    signature = signature.replace(
        "occupied housing units!!owner occupied",
        "owner occupied"
    )

    return signature


def get_column_label_by_year(report, column):
    labels_by_year = {}

    good = report[report["status"] == "OK"].copy()

    for _, row in good.iterrows():
        year = row["year"]
        labels = row["column_labels"]

        if isinstance(labels, dict) and column in labels:
            labels_by_year[year] = normalize_label(labels[column])

    return labels_by_year


def get_semantically_stable_columns(report):
    good = report[report["status"] == "OK"].copy()
    expected_years = sorted(good["year"].dropna().astype(int).unique().tolist())

    stable_cols = get_stable_columns(report)

    semantically_stable = []
    unstable = {}

    for col in stable_cols:
        labels_by_year = get_column_label_by_year(report, col)

        years_with_labels = sorted(
            pd.Series(list(labels_by_year.keys())).dropna().astype(int).tolist()
        )

        if years_with_labels != expected_years:
            unstable[col] = labels_by_year
            continue

        signatures = set(label_signature(label) for label in labels_by_year.values())

        if len(signatures) == 1:
            semantically_stable.append(col)
        else:
            unstable[col] = labels_by_year

    return semantically_stable, unstable

In [ ]:
def get_acs_column_labels(csv_file):
    labels_df = pd.read_csv(csv_file, dtype=str, nrows=1)
    return labels_df.iloc[0].to_dict()


def inspect_county_acs_folder(data_dir):
    data_files = get_data_files(data_dir)

    rows = []

    for f in data_files:
        year = get_year_from_filename(f)
        table = get_table_from_filename(f)

        try:
            labels = get_acs_column_labels(f)
            df = pd.read_csv(f, dtype=str, nrows=20)
        except Exception as e:
            rows.append({
                "file": f.name,
                "year": year,
                "table": table,
                "status": f"ERROR: {e}",
                "estimate_columns": [],
                "column_labels": {},
                "sample_values": {}
            })
            continue

        if "GEO_ID" in df.columns:
            df = df[df["GEO_ID"].ne("Geography")].copy()

        estimate_cols = [
            c for c in df.columns
            if c.endswith("E") and c not in ["GEO_ID", "NAME"]
        ]

        column_labels = {
            c: labels.get(c, "")
            for c in estimate_cols
        }

        sample_values = {}
        for col in estimate_cols[:20]:
            sample_values[col] = df[col].dropna().head(5).tolist()

        rows.append({
            "file": f.name,
            "year": year,
            "table": table,
            "status": "OK",
            "estimate_columns": estimate_cols,
            "column_labels": column_labels,
            "sample_values": sample_values
        })

    return pd.DataFrame(rows)


def summarize_county_acs_folder(data_dir):
    report = inspect_county_acs_folder(data_dir)

    print("Path:", data_dir)
    print("CSV files found:", len(report))

    good = report[report["status"] == "OK"].copy()

    years = sorted(good["year"].dropna().astype(int).unique().tolist())
    tables = sorted(good["table"].dropna().unique().tolist())

    print("\nYears found:")
    print(years)

    print("\nTables found:")
    print(tables)

    all_cols = []
    for cols in good["estimate_columns"]:
        all_cols.extend(cols)

    print("\nMost common estimate columns:")
    if all_cols:
        print(pd.Series(all_cols).value_counts().head(20))
    else:
        print("No estimate columns found.")

    semantically_stable_cols, unstable_cols = get_semantically_stable_columns(report)

    print("\nSemantically stable columns:")
    print(len(semantically_stable_cols))
    print(semantically_stable_cols[:40])

    print("\nUnstable columns:")
    print(len(unstable_cols))

    return report

In [ ]:
SQ_METERS_PER_SQ_MILE = 2589988.110336

def load_county_land_area(geography_path):
    geography_path = Path(geography_path)

    if not geography_path.exists():
        raise FileNotFoundError(f"Geography file not found: {geography_path}")

    counties = gpd.read_file(geography_path)

    required_cols = ["GEOID", "ALAND"]
    missing = [c for c in required_cols if c not in counties.columns]

    if missing:
        raise ValueError(f"Missing required columns in geography file: {missing}")

    area = counties[["GEOID", "ALAND"]].copy()

    area["county_fips"] = area["GEOID"].astype(str).str.zfill(5)
    area["ALAND"] = pd.to_numeric(area["ALAND"], errors="coerce")
    area["land_area_sq_mi"] = area["ALAND"] / SQ_METERS_PER_SQ_MILE

    area = area[area["land_area_sq_mi"].notna()].copy()
    area = area[area["land_area_sq_mi"] > 0].copy()

    return area[["county_fips", "ALAND", "land_area_sq_mi"]]

In [ ]:
def build_county_recipe_prompt(data_path, report, target_variable=None):
    good = report[report["status"] == "OK"].copy()

    target_text = ""
    if target_variable is not None:
        target_text = f"""
User requested variable:
{target_variable}
"""

    years = sorted(good["year"].dropna().astype(int).unique().tolist())
    tables = sorted(good["table"].dropna().unique().tolist())
    is_animated = len(years) > 1

    semantically_stable_cols, unstable_cols = get_semantically_stable_columns(report)
    common_cols = semantically_stable_cols[:80]

    stable_column_labels_text = "\nStable column labels:\n"

    if not good.empty:
        first_labels = good.iloc[0]["column_labels"]

        for col in common_cols:
            label = first_labels.get(col, "")
            stable_column_labels_text += f"- {col}: {label}\n"

    prompt = f"""
You are an AI assistant helping build animated Plotly county-level ACS maps.

Create one county map recipe for this ACS folder.

Folder:
{data_path}

{target_text}

Years found:
{years}

ACS tables found:
{tables}

Stable estimate columns with the same label every year:
{common_cols}

{stable_column_labels_text}

Dataset animation status:
{"animated" if is_animated else "static"}

The recipe will be used to create a whole-USA county choropleth animation.

Return only valid JSON with these keys:
project_title, geography_level, animated, acs_table, calculation, value_column, numerator_column, denominator_column, output_name, location_id, location_name, unit, color_scale, description, year_start, year_end.

Rules:
- geography_level must be "county".
- location_id should be "county_fips".
- location_name should be "county_name".
- If multiple years are found, set animated to true.
- If only one year is found, set animated to false.
- Pick value_column only from the stable estimate columns listed above.
- Use the same value_column for every year.
- Do not use year-specific columns.
- If no stable column correctly represents the requested variable, return "NEEDS_REVIEW" as value_column.
- Prioritize the user requested variable over the folder name.
- Prefer rates, percentages, or normalized values over raw counts when the dataset represents a subgroup of a larger total.
- Use raw counts only when the variable is naturally a total amount, such as total population, total households, median income, median value, or median age.
- If a column represents a subgroup and another column represents the total, set calculation to "percent".
- For percent calculations, set value_column to the numerator column, numerator_column to the subgroup column, denominator_column to the total column, and unit to "%".
- For direct values, set calculation to "direct", set numerator_column to null, and denominator_column to null.
- If the user requested unemployment, map unemployment rate, not employment rate.
- If the user requested employment, map employment rate, not unemployment rate.
- For B23025 unemployment rate, use unemployed divided by civilian labor force when both columns exist.
- For B23025 employment rate, use employed divided by civilian labor force when both columns exist.
- output_name must be lowercase snake case because it will become a dataframe column.
- color_scale should be a valid Plotly continuous color scale.
- Choose color_scale based on the meaning of the variable.
- Use "YlOrRd" or "OrRd" for risk, burden, poverty, cost, rent, unemployment, vulnerability, or other negative/higher-is-worse variables.
- Use "Blues" for water, flood, precipitation, or neutral count variables.
- Use "Greens" for income, home value, employment, access, resilience, or other positive/higher-is-better variables.
- Use "Purples" for age, population total, population density, or demographic intensity.
- Use "Cividis" if the variable is neutral and accessibility is preferred.
- Do not always choose Viridis.
- description should be one sentence.
- year_start and year_end should match the usable years found.
- Do not invent columns.
- If the user requests population density, use calculation "density".
- For population density, use total population as value_column and ALAND as denominator_column.
- For B01003 population density, use value_column "B01003_001E", denominator_column "ALAND", output_name "population_density", and unit "people/sq mi".
- Built-in county land area is available from the reference geography file using county_fips matched to GEOID.
- ALAND is land area in square meters and Python will convert it to square miles.
"""
    return prompt

In [ ]:
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def extract_json(text):
    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model response")

    return json.loads(match.group(0))


def generate_county_recipe_gemini(prompt):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt + "\n\nReturn only valid JSON. No markdown, no explanation."
    )

    return extract_json(response.text)

In [ ]:
def validate_county_recipe(recipe, report):
    good = report[report["status"] == "OK"].copy()

    all_cols = set()
    for cols in good["estimate_columns"]:
        all_cols.update(cols)

    semantically_stable_cols, unstable_cols = get_semantically_stable_columns(report)
    semantically_stable_cols = set(semantically_stable_cols)

    tables = set(good["table"].dropna().unique().tolist())
    years = sorted(good["year"].dropna().astype(int).unique().tolist())
    should_be_animated = len(years) > 1

    errors = []

    required_keys = [
        "project_title",
        "geography_level",
        "animated",
        "acs_table",
        "calculation",
        "value_column",
        "numerator_column",
        "denominator_column",
        "output_name",
        "location_id",
        "location_name",
        "unit",
        "color_scale",
        "description",
        "year_start",
        "year_end"
    ]

    for key in required_keys:
        if key not in recipe:
            errors.append(f"Missing key: {key}")

    if recipe.get("geography_level") != "county":
        errors.append("geography_level must be county")

    if recipe.get("acs_table") not in tables:
        errors.append(f"acs_table not found in files: {recipe.get('acs_table')}")

    if recipe.get("animated") != should_be_animated:
        errors.append(f"animated should be {should_be_animated} based on years found: {years}")

    if recipe.get("location_id") != "county_fips":
        errors.append("location_id should be county_fips")

    if recipe.get("location_name") != "county_name":
        errors.append("location_name should be county_name")

    if recipe.get("output_name"):
        if recipe["output_name"] != recipe["output_name"].lower():
            errors.append("output_name should be lowercase")
        if " " in recipe["output_name"]:
            errors.append("output_name should use underscores, not spaces")

    calculation = recipe.get("calculation")

    if calculation not in ["direct", "percent", "density"]:
        errors.append("calculation must be direct, percent, or density")

    if recipe.get("value_column") == "NEEDS_REVIEW":
        errors.append("AI could not find one stable column with the same meaning across all years.")

    elif calculation == "percent":
        numerator = recipe.get("numerator_column")
        denominator = recipe.get("denominator_column")

        if numerator not in all_cols:
            errors.append(f"numerator_column not found: {numerator}")

        if denominator not in all_cols:
            errors.append(f"denominator_column not found: {denominator}")

        if numerator not in semantically_stable_cols:
            errors.append(f"numerator_column is not semantically stable across all years: {numerator}")

        if denominator not in semantically_stable_cols:
            errors.append(f"denominator_column is not semantically stable across all years: {denominator}")

        if recipe.get("unit") != "%":
            errors.append("For percent calculations, unit should be %")

    elif calculation == "density":
        value_col = recipe.get("value_column")
        numerator = recipe.get("numerator_column")
        denominator = recipe.get("denominator_column")

        if value_col not in all_cols:
            errors.append(f"value_column not found: {value_col}")

        if value_col not in semantically_stable_cols:
            errors.append(f"value_column is not semantically stable across all years: {value_col}")

        if numerator is not None and numerator != value_col:
            errors.append("For density calculations, numerator_column should match value_column or be None")

        if denominator != "ALAND":
            errors.append("For density calculations, denominator_column should be ALAND")

        valid_density_units = [
            "people/sq mi",
            "people per sq mi",
            "people/square mile",
            "people per square mile"
        ]

        if recipe.get("unit") not in valid_density_units:
            errors.append("For population density, unit should be people/sq mi")

    elif calculation == "direct":
        value_col = recipe.get("value_column")

        if value_col not in all_cols:
            errors.append(f"value_column not found: {value_col}")

        if value_col not in semantically_stable_cols:
            errors.append(f"value_column is not semantically stable across all years: {value_col}")

        if recipe.get("numerator_column") is not None:
            errors.append("For direct calculations, numerator_column should be None")

        if recipe.get("denominator_column") is not None:
            errors.append("For direct calculations, denominator_column should be None")

    # Flexible selected years check
    if years:
        recipe_year_start = int(recipe.get("year_start", years[0]))
        recipe_year_end = int(recipe.get("year_end", years[-1]))

        if recipe_year_start < years[0]:
            errors.append(f"year_start cannot be earlier than first available year: {years[0]}")

        if recipe_year_end > years[-1]:
            errors.append(f"year_end cannot be later than last available year: {years[-1]}")

        if recipe_year_start > recipe_year_end:
            errors.append("year_start cannot be greater than year_end")

    if errors:
        print("Recipe has problems:")
        for e in errors:
            print("-", e)

        print("\nStable columns available:")
        print(sorted(list(semantically_stable_cols))[:50])

        print("\nSome unstable columns:")
        for col, labels in list(unstable_cols.items())[:5]:
            print("\n", col)
            for year, label in labels.items():
                print(year, ":", label)

        return False

    print("Recipe passed validation.")
    return True

In [ ]:
def build_county_long_from_recipe(data_dir, recipe, geography_path=None):
    data_dir = Path(data_dir)

    year_start = int(recipe.get("year_start", 2011))
    year_end = int(recipe.get("year_end", 2024))

    calculation = recipe.get("calculation")
    output_col = recipe["output_name"]

    area_lookup = None

    if calculation == "density":
        if geography_path is None:
            raise ValueError("Density calculation requires geography_path with county ALAND land area.")

        area_lookup = load_county_land_area(geography_path)

    records = []

    for data_path in get_data_files(data_dir):
        year = get_year_from_filename(data_path)

        if year is None or year < year_start or year > year_end:
            continue

        df = pd.read_csv(data_path, dtype=str)

        if "GEO_ID" not in df.columns:
            print(f"Skipping {data_path.name}: no GEO_ID column")
            continue

        df = df[df["GEO_ID"].ne("Geography")].copy()

        df["county_fips"] = county_fips_from_geo_id(df["GEO_ID"])
        df = df[df["county_fips"].notna()].copy()
        df["county_fips"] = df["county_fips"].astype(str).str.zfill(5)

        if calculation == "percent":
            numerator_col = recipe["numerator_column"]
            denominator_col = recipe["denominator_column"]

            if numerator_col not in df.columns or denominator_col not in df.columns:
                print(f"Skipping {year}: missing numerator or denominator")
                continue

            numerator = df[numerator_col].apply(clean_numeric_value)
            denominator = df[denominator_col].apply(clean_numeric_value)

            df[output_col] = (numerator / denominator) * 100
            df.loc[denominator <= 0, output_col] = pd.NA

        elif calculation == "density":
            value_col = recipe["value_column"]

            if value_col not in df.columns:
                print(f"Skipping {year}: missing column {value_col}")
                continue

            df[value_col] = df[value_col].apply(clean_numeric_value)

            df = df.merge(area_lookup, on="county_fips", how="left")

            df[output_col] = df[value_col] / df["land_area_sq_mi"]
            df.loc[df["land_area_sq_mi"] <= 0, output_col] = pd.NA

        else:
            value_col = recipe["value_column"]

            if value_col not in df.columns:
                print(f"Skipping {year}: missing column {value_col}")
                continue

            df[output_col] = df[value_col].apply(clean_numeric_value)

        df["year"] = year
        df["county_name"] = df["NAME"]

        records.append(
            df[["county_fips", "county_name", "year", output_col]]
        )

    if not records:
        raise ValueError("No data loaded. Check recipe and folder.")

    county_long = pd.concat(records, ignore_index=True)

    county_long["year"] = county_long["year"].astype(int)
    county_long["county_fips"] = county_long["county_fips"].astype(str).str.zfill(5)
    county_long[output_col] = pd.to_numeric(
        county_long[output_col],
        errors="coerce"
    )

    return county_long

In [ ]:
def get_auto_range_color(plot_data, value_col, recipe):
    values = plot_data[value_col].dropna()

    if values.empty:
        return None

    calculation = str(recipe.get("calculation", "")).lower()
    unit = str(recipe.get("unit", "")).lower()
    output_name = str(recipe.get("output_name", "")).lower()
    title = str(recipe.get("project_title", "")).lower()

    text = f"{output_name} {title} {unit}"

    # Percent/rate maps are usually best with a slightly tighter range
    if unit == "%" or calculation == "percent" or "rate" in text or "percent" in text or "pct" in text:
        low = values.quantile(0.02)
        high = values.quantile(0.98)

    # Density, population, income, home values, rent often have big outliers
    elif (
        calculation == "density"
        or "density" in text
        or "population" in text
        or "income" in text
        or "value" in text
        or "rent" in text
    ):
        low = values.quantile(0.05)
        high = values.quantile(0.95)

    # General fallback
    else:
        low = values.quantile(0.02)
        high = values.quantile(0.98)

    if pd.isna(low) or pd.isna(high) or low == high:
        return None

    return [low, high]

In [ ]:
def make_county_animation_map(county_long, recipe):
    value_col = recipe["output_name"]

    plot_data = county_long.dropna(
        subset=["year", "county_fips", value_col]
    ).copy()

    plot_data["year"] = plot_data["year"].astype(int).astype(str)
    plot_data["county_fips"] = plot_data["county_fips"].astype(str).str.zfill(5)

    counties_geojson = requests.get(
        "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
    ).json()

    unit = recipe.get("unit", "")
    
    if unit:
        colorbar_label = unit
    else:
        colorbar_label = value_col.replace("_", " ").title()
    
    fig = px.choropleth(
        plot_data,
        geojson=counties_geojson,
        locations="county_fips",
        color=value_col,
        animation_frame="year",
        animation_group="county_fips",
        scope="usa",
        hover_name="county_name",
        hover_data={
            "county_fips": True,
            value_col: ":,.1f" if recipe.get("unit") in ["%", "people/sq mi"] else ":,.0f",
            "year": True
        },
        labels={
            value_col: colorbar_label,
            "county_fips": "County FIPS",
            "year": "Year"
        },
        color_continuous_scale=recipe.get("color_scale", "Viridis"),
        range_color=get_auto_range_color(plot_data, value_col, recipe)
    )

    fig.update_layout(
        width=1200,
        height=800,
        margin={"r": 0, "t": 70, "l": 0, "b": 80},
        title={
            "text": f'{recipe["project_title"]}, {recipe["year_start"]}–{recipe["year_end"]}',
            "font": {"size": 24}
        },
        sliders=[{
            "currentvalue": {
                "font": {"size": 24},
                "prefix": "Year: ",
                "visible": True
            },
            "font": {"size": 18},
            "len": 0.85,
            "x": 0.08,
            "y": 0,
            "pad": {"b": 10, "t": 40},
            "ticklen": 8
        }],
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 0.02,
            "y": 0,
            "xanchor": "right",
            "yanchor": "top",
            "pad": {"r": 10, "t": 35},
            "font": {"size": 18},
            "buttons": [{
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 700, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 300}
                    }
                ]
            }, {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }
                ]
            }]
        }]
    )

    return fig

In [ ]:
from pathlib import Path

def get_data_files(data_path):
    data_path = Path(data_path)

    if data_path.is_file():
        return [data_path]

    if data_path.is_dir():
        # First try normal ACS data files
        data_files = sorted(data_path.rglob("*-Data.csv"))

        # Fallback: if none found, grab all CSVs
        if len(data_files) == 0:
            data_files = sorted(data_path.rglob("*.csv"))

        return data_files

    raise FileNotFoundError(f"Path not found: {data_path}")

In [ ]:
from pathlib import Path
from datetime import datetime

def make_export_path(recipe, output_folder=r"C:\Users\samoj\Downloads"):
    output_folder = Path(output_folder)

    safe_name = recipe["output_name"]
    year_start = recipe["year_start"]
    year_end = recipe["year_end"]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    filename = f"{safe_name}_{year_start}_{year_end}_{timestamp}.html"

    return output_folder / filename

In [ ]:
CACHE_DIR = Path(r"C:\Users\samoj\Downloads\MapAnimations\Cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def make_cache_key(*items):
    """
    Makes a stable cache filename from paths, recipe settings, target variable, etc.
    """
    text = json.dumps([str(x) for x in items], sort_keys=True)
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def cache_load(cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    return None


def cache_save(obj, cache_path):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump(obj, f)

In [ ]:
def run_county_animation_ai(
    data_path,
    target_variable=None,
    geography_path=None,
    year_start=None,
    year_end=None,
    show_prompt_preview=True,
    prompt_preview_chars=2000,
    stop_on_validation_error=True,
    export_folder=None,
    use_cache=True,
    refresh_recipe=False,
    refresh_long=False
):
    data_path = Path(data_path)

    if geography_path is None:
        geography_path = DEFAULT_COUNTY_GEOGRAPHY_PATH

    print("Using data path:")
    print(data_path)

    print("\nChecking path...")
    print("Exists:", data_path.exists())
    print("Is folder:", data_path.is_dir())
    print("Is file:", data_path.is_file())

    if not data_path.exists():
        raise FileNotFoundError(f"Path not found: {data_path}")

    print("\nFinding CSV files...")
    data_files = get_data_files(data_path)

    print("CSV files found:", len(data_files))
    for f in data_files[:20]:
        print("-", f.name)

    if len(data_files) == 0:
        raise ValueError("No CSV files found. Check your folder path or file names.")

    print("\nInspecting county ACS source...")
    report = summarize_county_acs_folder(data_path)

    print("\nBuilding Gemini prompt...")
    prompt = build_county_recipe_prompt(
        data_path=data_path,
        report=report,
        target_variable=target_variable
    )

    if show_prompt_preview:
        print("\nPrompt preview:")
        print(prompt[:prompt_preview_chars])

    recipe_cache_key = make_cache_key(
        "recipe",
        data_path,
        target_variable,
        year_start,
        year_end
    )

    recipe_cache_path = CACHE_DIR / f"{recipe_cache_key}_recipe.pkl"

    county_recipe = None

    if use_cache and not refresh_recipe:
        county_recipe = cache_load(recipe_cache_path)

    if county_recipe is not None:
        print("\nLoaded cached Gemini recipe.")
    else:
        print("\nGenerating county recipe with Gemini...")
        county_recipe = generate_county_recipe_gemini(prompt)

        if use_cache:
            cache_save(county_recipe, recipe_cache_path)
            print("Saved recipe to cache.")

    # Optional override so you can ignore 2010 or choose a smaller range
    if year_start is not None:
        county_recipe["year_start"] = year_start

    if year_end is not None:
        county_recipe["year_end"] = year_end

    print("\nGenerated recipe:")
    display(county_recipe)

    print("\nValidating recipe...")
    recipe_ok = validate_county_recipe(county_recipe, report)

    if not recipe_ok:
        if stop_on_validation_error:
            raise ValueError("Recipe validation failed. Fix the recipe before building the map.")
        else:
            print("Validation failed, but continuing because stop_on_validation_error=False.")

    long_cache_key = make_cache_key(
        "county_long",
        data_path,
        geography_path,
        county_recipe
    )

    long_cache_path = CACHE_DIR / f"{long_cache_key}_county_long.pkl"

    county_long = None

    if use_cache and not refresh_long:
        county_long = cache_load(long_cache_path)

    if county_long is not None:
        print("\nLoaded cached county long-format data.")
    else:
        print("\nBuilding county long-format data...")
        county_long = build_county_long_from_recipe(
            data_path,
            county_recipe,
            geography_path=geography_path
        )

        if use_cache:
            cache_save(county_long, long_cache_path)
            print("Saved county long-format data to cache.")

    print("County long data shape:", county_long.shape)

    print("\nYears included:")
    print(county_long["year"].value_counts().sort_index())

    print("\nPreview:")
    display(county_long.head())

    print("\nMaking county map...")
    fig = make_county_animation_map(county_long, county_recipe)
    fig.show()

    if export_folder is not None:
        export_path = make_export_path(
            county_recipe,
            output_folder=export_folder
        )

        fig.write_html(
            export_path,
            include_plotlyjs="cdn",
            full_html=True
        )

        print("Saved animation to:", export_path)

    print("\nDone.")

    return county_recipe, county_long, fig

# Testing

In [ ]:
#Income
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\MedianIncome_2011-2024_County_2",
    year_start=2011,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Median Land Value
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\MedianLandValue_2010-2024_County",
    year_start=2011,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Poverty Rate
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\PovertyRate2010-2024_County",
    year_start=2012,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Total Population
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\PopulationTotals_2010-2024_County",
    year_start=2012,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Employment
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\EmploymentRate_2011-2024_County",
    year_start=2012,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Unemployment (Calculation Test)
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\EmploymentRate_2011-2024_County",
    target_variable="unemployment rate",
    year_start=2012,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Percent Renter (Calculation Test)
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\HousingTenure_2010-2024_County",
    target_variable="percent renter occupied housing units",
    year_start=2011,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Population Density
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\PopulationTotals_2010-2024_County",
    target_variable="population density",
    year_start=2012,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Cache Testing 

In [ ]:

county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\HousingTenure_2010-2024_County",
    target_variable="percent renter occupied housing units",
    year_start=2011,
    year_end=2024,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)

In [ ]:
# Percent Renter (Calculation Test)
county_recipe, county_long, fig = run_county_animation_ai(
    data_path=r"C:\Users\samoj\Downloads\SCIL_Data\Socioeconomic_AnimationData\HousingTenure_2010-2024_County",
    target_variable="percent renter occupied housing units",
    year_start=2011,
    year_end=2024,
    refresh_long=True,
    export_folder=r"C:\Users\samoj\Downloads\MapAnimations"
)